In [ ]:
from config_dirs import PERPLEXITY_INPUT, STANDARDIZED_GENES_OUTPUT

import sys
print(sys.version)
import openai
print(openai.__version__)
import pandas as pd
import requests
import os
from openai import OpenAI

In [ ]:
pd.read_csv(PERPLEXITY_INPUT).query("`Entity of Association` == 'Protein'").to_excel('OpenAI_Output_Protein_only.xlsx', index=False)

file_name = 'OpenAI_Output_Protein_only.xlsx'
import pandas as pd
# Load the Excel file into a pandas DataFrame
df = pd.read_excel(file_name)



In [ ]:
import time
import pandas as pd
import openai
from openai import OpenAI
# Initialize OpenAI client
YOUR_API_KEY = "your_api_key_here"
client = openai.OpenAI(api_key=YOUR_API_KEY, base_url="https://api.perplexity.ai")

# Function to query the LLM for the standardized gene/protein name
def get_standardized_name(gene_name, cache):
    if gene_name in cache:
        return cache[gene_name]
    
    messages = [
        {
            "role": "system",
            "content": (
                "You are an AI assistant with expert-level knowledge in genomics and molecular biology. "
                "Please return only the Gene Symbol based on the HUGO Gene Nomenclature Committee (HGNC) guidelines without any additional text or explanation."
            ),
        },
        {
            "role": "user",
            "content": f"{gene_name}",
        },
    ]

    # Perform the API call
    response = client.chat.completions.create(
        model="llama-3.1-sonar-huge-128k-online",
        messages=messages,
    )
    
    # Extract and return the standardized name from the response
    standardized_name = response.choices[0].message.content
    
    # Cache the result
    cache[gene_name] = standardized_name
    
    return standardized_name

# Function to process the DataFrame with rate limiting, caching, and retry logic
def process_dataframe(df, save_interval=20, max_retries=3):
    request_count = 0
    processed_count = 0
    cache = {}
    
    for index, row in df.iterrows():
        gene_name = row['Name']
        
        for retry in range(max_retries):
            try:
                standardized_name = get_standardized_name(gene_name, cache)
                df.at[index, 'Perplexity'] = standardized_name
                print(f"Processed row {index}: {gene_name} -> {standardized_name}")
                
                # Increment the request count only if it's not retrieved from cache
                if gene_name not in cache:
                    request_count += 1
                
                processed_count += 1
                
                # Check if rate limit is close to being exceeded
                if request_count >= 20:
                    # Save progress to a file
                    df.to_csv('standardized_genes_partial.csv', index=False)
                    print(f"Reached rate limit, saved progress after processing {processed_count} rows.")
                    
                    # Reset request count and sleep for 60 seconds
                    request_count = 0
                    time.sleep(60)
                
                break  # Exit retry loop if successful
            except Exception as e:
                if "rate limit" in str(e).lower():
                    if retry < max_retries - 1:
                        print(f"Rate limit exceeded. Retrying in 60 seconds...")
                        time.sleep(60)
                    else:
                        raise
                else:
                    print(f"Error processing row {index}: {e}")
                    break  # Exit retry loop if there's an error other than rate limit
        
        # Save progress periodically
        if processed_count % save_interval == 0:
            df.to_csv('standardized_genes_partial.csv', index=False)
            print(f"Saved progress after processing {processed_count} rows.")
    
    # Final save after all rows are processed
    df.to_csv(STANDARDIZED_GENES_OUTPUT, index=False)
    print("Finished processing all rows.")

# Example setup for the DataFrame (this should be your actual DataFrame)
# df = pd.read_csv('your_data.csv')  # Load your actual data here

# Initialize a 'Perplexity' column in the DataFrame
df['Perplexity'] = None



In [ ]:
process_dataframe(df)